# Semana 14: Cápsulas: texto y series de tiempo

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
s = pd.read_csv(BASE + "frases_sentimiento.csv")
print(s.shape, s["fuente"].value_counts().to_dict(), " positivas:", s["sentimiento"].sum())

## Kit 1: texto
Demo base: bolsa de palabras y regresión logística.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(s["frase"], s["sentimiento"], test_size=0.25, random_state=42, stratify=s["sentimiento"])
v = CountVectorizer(); m = LogisticRegression(max_iter=2000).fit(v.fit_transform(Xtr), ytr)
p = m.predict(v.transform(Xte))
print("Vocabulario", len(v.vocabulary_), " exactitud", round(m.score(v.transform(Xte), yte), 3), " matriz", confusion_matrix(yte, p).tolist())
co = pd.Series(m.coef_[0], index=v.get_feature_names_out())
print("Más positivas:", co.nlargest(5).round(2).to_dict()); print("Más negativas:", co.nsmallest(5).round(2).to_dict())
sondas = ["great", "not good", "not bad", "the battery is not bad at all"]
print({t: round(m.predict_proba(v.transform([t]))[0, 1], 3) for t in sondas})

### Mejora: bigramas. ¿Arregla la negación?

In [ ]:
v2 = CountVectorizer(ngram_range=(1, 2)); m2 = LogisticRegression(max_iter=2000).fit(v2.fit_transform(Xtr), ytr)
print("Términos", len(v2.vocabulary_), " exactitud", round(m2.score(v2.transform(Xte), yte), 3))
print({t: round(m2.predict_proba(v2.transform([t]))[0, 1], 3) for t in sondas})
print("Frases con 'not bad':", s["frase"].str.lower().str.contains("not bad", regex=False).sum())

### Cambio de dominio: entrenar con Amazon y Yelp, probar con IMDb

In [ ]:
fuera = s["fuente"] != "IMDb"
v3 = CountVectorizer(); m3 = LogisticRegression(max_iter=2000).fit(v3.fit_transform(s["frase"][fuera]), s["sentimiento"][fuera])
print("Exactitud en IMDb:", round(m3.score(v3.transform(s["frase"][~fuera]), s["sentimiento"][~fuera]), 3))

## Kit 2: series de tiempo

In [ ]:
h = pd.read_csv(BASE + "bicicletas_hora.csv")
perfil = h.groupby(["workingday", "hr"])["cnt"].mean().unstack(0)
print("Filas:", len(h), " picos laboral:", perfil[1].nlargest(2).index.tolist(), " no laboral:", perfil[0].nlargest(2).index.tolist())
perfil.plot(title="Dos ciudades en una"); plt.xlabel("Hora"); plt.show()
d = pd.read_csv(BASE + "bicicletas_dia.csv", parse_dates=["dteday"]).set_index("dteday")["cnt"]
print("Autocorrelación 1 día", round(d.autocorr(1), 3), " 7 días", round(d.autocorr(7), 3), " 7 días con diferencias", round(d.diff().autocorr(7), 3))
d.plot(alpha=.4); d.rolling(7, center=True).mean().plot(title="Media móvil de 7 días"); plt.show()

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.